# Table wiping — Day 1: capture → synchronize → the invisible made visible

**MIT Professional Education · Mastering Integrated Systems**

AI is remarkable at reading and reshaping the *appearance* of the physical world.
The next step is understanding how physical things — humans included — *behave and
move*, and we build that understanding through **multimodal sensing and measurement**.

Here you have one recording of a hand wiping a table, captured with **motion capture**,
**EMG** (forearm flexors + extensors), and **forearm ultrasound** — already synchronized
onto a single clock (the OptiTrack timebase). Everything in this notebook reads from one
small bundle file; nothing here needs the lab's acquisition stack.

> Run-all works: `Runtime → Run all`. Then come back and play with the zoom + the bars.

## Setup — point at the dataset bundle

In [ ]:
#@title Setup { run: "auto" }
import os
BUNDLE_DIR = ""  #@param {type:"string"}
# In Colab: download + unzip the shared bundle, then set BUNDLE_DIR to its folder, e.g.
#   !gdown <file-id> -O bundle.zip && unzip -q bundle.zip
if not BUNDLE_DIR:
    for c in ("_out/bundle", "../_out/bundle", os.environ.get("ILPEMIS_BUNDLE", ""), "."):
        if c and os.path.exists(os.path.join(c, "table_wiping.h5")):
            BUNDLE_DIR = c; break
H5 = os.path.join(BUNDLE_DIR, "table_wiping.h5")
assert os.path.exists(H5), f"bundle not found - set BUNDLE_DIR (looked under {BUNDLE_DIR!r})"
print("bundle:", os.path.abspath(H5))

## Load the aligned record\nOne HDF5, every modality already on the OptiTrack clock (0 = first mocap frame).

In [ ]:
import h5py, numpy as np, matplotlib.pyplot as plt

def load_bundle(path):
    def visit(g):
        d = {k: (visit(v) if isinstance(v, h5py.Group) else v[()]) for k, v in g.items()}
        d.update({"@" + k: v for k, v in g.attrs.items()})
        return d
    with h5py.File(path, "r") as h:
        return visit(h)

B = load_bundle(H5)
SEG = {k[1:]: tuple(v) for k, v in B["segments"].items() if k.startswith("@")}
print("modalities:", [k for k in B if not k.startswith("@")])
print("EMG channels:", list(B["emg"]))
print("ultrasound:", round(float(B["us"]["@fps"]), 1), "fps,", int(B["us"]["@n_frames"]), "frames")
print("segments (OT s):", {k: (round(a, 1), round(b, 1)) for k, (a, b) in SEG.items()})

## See — three modalities, one clock

Flexor EMG (muscle activation), ultrasound tissue **deformation** — the distance between
two tracked points *inside* the forearm — and hand speed (the visible movement). The shaded
bands are the task conditions **Normal / Tensed / Slow / Fast**. Notice each signal goes
quiet between bouts — that's the synchronization working: they rise and fall together.

In [ ]:
SEG_COLORS = {"Normal": "#3a7d44", "Tensed": "#b03a3a", "Slow": "#999999", "Fast": "#2f6f9f"}
us = B["us"]
US_V = us["distance"] if "distance" in us else us["motion_proxy"]
US_T = us["frame_times_ot"][:len(US_V)]
US_LBL = "US tissue distance\n(px)" if "distance" in us else "US tissue\n(frame-diff)"

def panels(xlim=None, shade=True, title=""):
    flx, mo = B["emg"]["RForearmFlexors"], B["mocap"]
    rows = [("flexor EMG\n(mV)", flx["t_ot"], flx["rms_mV"], "#b03a3a"),
            (US_LBL, US_T, US_V, "#7a3fa0"),
            ("hand speed\n(mm/s)", mo["t_ot"], mo["hand_speed_mm_s"], "#333333")]
    if xlim is None:
        xlim = (0.0, float(np.nanmax(US_T)))
    fig, ax = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
    for a, (lbl, t, v, c) in zip(ax, rows):
        t, v = np.asarray(t), np.asarray(v)
        a.plot(t, v, color=c, lw=0.6); a.set_ylabel(lbl); a.grid(alpha=0.2); a.set_xlim(xlim)
        w = (t >= xlim[0]) & (t <= xlim[1])     # scale y to the visible window
        if w.any():
            lo, hi = np.nanpercentile(v[w], [1, 99]); pad = 0.08 * (hi - lo + 1e-9)
            a.set_ylim(lo - pad, hi + pad)
        if shade:
            for k, (x0, x1) in SEG.items():
                a.axvspan(x0, x1, color=SEG_COLORS[k], alpha=0.10)
    ax[-1].set_xlabel("OptiTrack time (s)"); ax[0].set_title(title)
    plt.tight_layout(); plt.show()

panels(title="table wiping - three modalities, one clock")

## Zoom in — the cascade

Tighten the window to a few strokes. Within each wipe you can see the **chain**: the muscle
fires (EMG), the tissue moves (ultrasound), and the hand moves (speed). Change the numbers
and re-run.

In [ ]:
#@title Zoom into the cascade
zoom_start = 14  #@param {type:"number"}
zoom_end = 19  #@param {type:"number"}
panels(xlim=(zoom_start, zoom_end), shade=False,
       title=f"EMG -> tissue -> hand, {zoom_start}-{zoom_end} s")

## Going deeper — how the ultrasound joined this clock

The ultrasound ran at ~133 fps on *its own* device clock. To put every frame on the
OptiTrack timeline, the probe's **FrameOutput** pulse for each frame was recorded on the
same box as the EMG, then mapped through the recording-gate. Those pulses are rock-steady —
a far better per-frame clock than the file's own timestamps.

In [ ]:
ft = np.asarray(B["us"]["frame_times_ot"]); isi = np.diff(ft) * 1000.0
print(f"US frames: {len(ft)} | mean inter-frame {isi.mean():.2f} ms "
      f"({1000/isi.mean():.1f} fps) | jitter (std) {isi.std():.3f} ms")
plt.figure(figsize=(6, 3)); plt.hist(isi, bins=60, color="#7a3fa0")
plt.xlabel("inter-frame interval (ms)"); plt.ylabel("count")
plt.title("per-frame ultrasound clock (FrameOutput c-spikes)"); plt.tight_layout(); plt.show()

## The two contrasts

**Normal vs Tensed** — same movement, but bracing: *both* muscles fire harder
(co-contraction). **Normal vs Fast** — faster strokes, higher hand speed.

In [ ]:
def seg_mean(t, v, k):
    a, b = SEG[k]; m = (np.asarray(t) >= a) & (np.asarray(t) <= b)
    return float(np.nanmean(np.asarray(v)[m]))

conds = ["Normal", "Tensed", "Fast"]
flx, ext, mo = B["emg"]["RForearmFlexors"], B["emg"]["RForearmExtensors"], B["mocap"]
fe = [seg_mean(flx["t_ot"], flx["rms_mV"], k) for k in conds]
ee = [seg_mean(ext["t_ot"], ext["rms_mV"], k) for k in conds]
sp = [seg_mean(mo["t_ot"], mo["hand_speed_mm_s"], k) for k in conds]
x = np.arange(len(conds))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(x - 0.18, fe, 0.36, label="flexor", color="#b03a3a")
ax[0].bar(x + 0.18, ee, 0.36, label="extensor", color="#2f6f9f")
ax[0].set_xticks(x); ax[0].set_xticklabels(conds); ax[0].set_ylabel("EMG RMS (mV)")
ax[0].legend(); ax[0].set_title("Normal vs Tensed = co-contraction")
ax[1].bar(x, sp, 0.6, color=[SEG_COLORS[k] for k in conds])
ax[1].set_xticks(x); ax[1].set_xticklabels(conds); ax[1].set_ylabel("hand speed (mm/s)")
ax[1].set_title("Normal vs Fast")
plt.tight_layout(); plt.show()
print("Tensed/Normal flexor EMG x", round(fe[1] / fe[0], 2),
      "| Fast/Normal hand speed x", round(sp[2] / sp[0], 2))

## Your turn

You have `B` (the loaded bundle) with `emg`, `mocap`, `us`, and `segments`. Try, or ask the
assistant to help you:
- overlay the **extensor** EMG on the zoom and compare flexor vs extensor timing within a stroke;
- compute the **lag** between an EMG burst and the hand-speed peak;
- plot the raw tracked tissue points `B['us']['track']['point0' / 'point1']` (x, y per frame);
- pick your own time window and a different marker from `B['mocap']['markers']`.

Tomorrow you'll *track* the ultrasound tissue itself (DUSTrack) and design your own capture.